# 온기 DEV-08 — 임베딩 판단 기준 만들기 (Colab)

키워드로 못 잡은 아이 말을 **기준 문장과 비교**해서 상황을 정한다. 이 노트북이 만드는 것:

1. 기준 문장 벡터 `anchors.f32` + `anchors_meta.json` (학습 데이터 v3.5.2 전체, 일반 대화·상황 불명확은 0=없음)
2. **임계값**: 평가 290문장에서 "키워드 → 임베딩"을 합쳐 잘못 재생 5% 이하에서 정답이 가장 많은 값
3. 보드용 모델: `minilm_int8.onnx`(토큰 → 384차원, 평균·정규화까지 안에 포함) + `vocab.tsv`(토크나이저 단어장)
   → 토크나이저는 보드 앱(Kotlin)이 직접 계산(Unigram). 여기서 같은 방식을 파이썬으로 돌려 원래 토크나이저와 100% 같은지 확인

**시작 전**: GPU 필요 없음(CPU로 충분). 위에서부터 차례로 실행, 약 10~15분.


In [ ]:
# 1. 설치
!pip -q install -U sentence-transformers onnx onnxruntime openpyxl

In [ ]:
# 2. 파일 올리기 (2개)
#    00_온기_학습데이터_v3.5.2.xlsx  (ai/data)
#    situations.json                  (device/android/app/src/main/assets)
import glob, os, unicodedata
N = lambda f: unicodedata.normalize("NFC", f)
need = lambda: [f for f in glob.glob("/content/*.xlsx") if "v3." in N(f) and "참고자료" not in N(f)] and os.path.exists("/content/situations.json")
if not need():
    from google.colab import files
    os.chdir("/content"); files.upload()
print("준비됨" if need() else "아직 빠진 파일이 있어요")

In [ ]:
# 3. 데이터 읽기 + 키워드 판단 (기기 KeywordJudge와 같은 방식)
import json, re, numpy as np, pandas as pd
XLSX = sorted([f for f in glob.glob("/content/*.xlsx") if "v3." in N(f) and "참고자료" not in N(f)])[-1]
P = re.compile(r"<maum_(\d)>\((?:situation=(-?\d+), feeling=(-?\d+))?\)")
def gold(o):
    m = P.search(str(o)); return 0 if m.group(1) == "1" else int(m.group(2))
def load(sheet):
    d = pd.read_excel(XLSX, sheet_name=sheet)
    return pd.DataFrame({"text": d["Query(한글)"].astype(str), "gold": d["LLM Output"].map(gold)})
train, evals = load("Function_Data_싱글턴"), load("Eval_Set")
cfg = json.load(open("/content/situations.json", encoding="utf-8"))
sq = lambda s: s.replace(" ", "")
RULES = [dict(id=s["id"], kw=[sq(k) for k in s["keywords"]], ex=[sq(k) for k in s["exclude"]], bl=[sq(k) for k in s["block"]]) for s in cfg["situations"]]
def keyword(text):
    base = sq(text); hit = set()
    for r in RULES:
        if any(b in base for b in r["bl"]): continue
        t = base
        for e in r["ex"]: t = t.replace(e, " ")
        if any(k in t for k in r["kw"]): hit.add(r["id"])
    order = [p for p in cfg["priority"] if p in hit]
    return order[0] if order else None
for d in (train, evals): d["kw"] = d["text"].map(keyword)
print(os.path.basename(XLSX), "/ 학습", len(train), "/ 평가", len(evals))
e_situ = evals[evals.gold > 0]
print(f"평가 키워드: 상황 맞힘 {(e_situ.kw == e_situ.gold).mean()*100:.1f}% / 못 잡음 {e_situ.kw.isna().sum()} → 임베딩으로")

In [ ]:
# 4. MiniLM으로 기준 문장 벡터 (상황 발화 → 그 번호, 일반 대화·상황 불명확 → 0)
from sentence_transformers import SentenceTransformer
MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
st = SentenceTransformer(MODEL, device="cpu")
anchor_text = train["text"].tolist()
anchor_id = [g if g > 0 else 0 for g in train["gold"]]
A = st.encode(anchor_text, normalize_embeddings=True, batch_size=64, show_progress_bar=True).astype(np.float32)
E = st.encode(evals["text"].tolist(), normalize_embeddings=True, batch_size=64).astype(np.float32)
print("기준 문장", A.shape, "/ 상황별 수:", pd.Series(anchor_id).value_counts().sort_index().to_dict())

In [ ]:
# 5. 임계값 찾기 — 키워드 먼저, 없으면 임베딩(가장 가까운 기준 문장이 0이 아니고 점수 ≥ 임계값일 때만)
anchor_id_arr = np.array(anchor_id)
def embed_pick(vecs, k=1):
    sims = vecs @ A.T
    top = np.argsort(-sims, axis=1)[:, :k]
    best = top[:, 0]
    return anchor_id_arr[best], sims[np.arange(len(vecs)), best]
emb_id, emb_score = embed_pick(E)
evals["emb_id"], evals["emb_score"] = emb_id, emb_score

def combined(th):
    pred = []
    for kw, eid, sc in zip(evals.kw, evals.emb_id, evals.emb_score):
        if kw is not None and not pd.isna(kw): pred.append(int(kw))
        elif eid != 0 and sc >= th: pred.append(int(eid))
        else: pred.append(None)
    return pred

rows = []
situ = evals.gold > 0
for th in np.arange(0.40, 0.96, 0.05):
    p = pd.Series(combined(th), index=evals.index, dtype="object")
    right = (p[situ] == evals.gold[situ]).sum()
    wrong = (p[situ].notna() & (p[situ] != evals.gold[situ])).sum()
    false = p[~situ].notna().sum()
    rows.append(dict(임계값=round(th, 2), 상황맞힘=round(right / situ.sum() * 100, 1), 다른상황=int(wrong),
                     일반잘못잡음=int(false), 잘못재생률=round((wrong + false) / len(evals) * 100, 1)))
sweep = pd.DataFrame(rows); print(sweep.to_string(index=False))
ok = sweep[sweep.잘못재생률 <= 5.0]
best = ok.sort_values(["상황맞힘", "임계값"], ascending=[False, False]).iloc[0] if len(ok) else sweep.iloc[-1]
TH = float(best.임계값)
print(f"\n선택 임계값 {TH}: 상황 맞힘 {best.상황맞힘}% / 다른 상황 {best.다른상황} / 일반 잘못 잡음 {best.일반잘못잡음}")

In [ ]:
# 6. 임베딩으로 판단한 문장 확인 (키워드가 못 잡은 평가 문장)
p = combined(TH)
chk = evals.assign(pred=p)[evals.kw.isna()][["text", "gold", "emb_id", "emb_score", "pred"]]
chk["맞음"] = [(g > 0 and pr == g) or (g <= 0 and pr is None) for g, pr in zip(chk.gold, chk.pred)]
print(f"키워드가 못 잡은 {len(chk)}문장 중 임베딩 결과 맞음 {chk.맞음.sum()}")
print(chk.sort_values("맞음").to_string(index=False))

In [ ]:
# 7. 보드용 ONNX — 모델(평균·정규화 포함) + 토크나이저
import torch, onnx, onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType
OUT = "/content/ongi_embed"; os.makedirs(OUT, exist_ok=True)
hf = st[0].auto_model.eval(); tok = st.tokenizer

class Wrap(torch.nn.Module):
    def __init__(s, m): super().__init__(); s.m = m
    def forward(s, input_ids, attention_mask):
        h = s.m(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        v = (h * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
        return torch.nn.functional.normalize(v, dim=-1)
ex = tok(["오늘 친구랑 싸웠어"], return_tensors="pt")
torch.onnx.export(Wrap(hf), (ex["input_ids"], ex["attention_mask"]), f"{OUT}/minilm.onnx",
                  input_names=["input_ids", "attention_mask"], output_names=["embedding"],
                  dynamic_axes={"input_ids": {0: "b", 1: "t"}, "attention_mask": {0: "b", 1: "t"}, "embedding": {0: "b"}},
                  opset_version=17, do_constant_folding=True, dynamo=False)   # 새 torch는 기본이 dynamo(onnxscript 필요) → 예전 방식
quantize_dynamic(f"{OUT}/minilm.onnx", f"{OUT}/minilm_int8.onnx", weight_type=QuantType.QInt8)

# 토크나이저: ONNX 변환 대신 단어장(vocab.tsv)을 저장하고, 보드 앱이 같은 계산(Unigram)을 직접 한다
tj = json.loads(tok.backend_tokenizer.to_str())
assert tj["model"]["type"] == "Unigram", tj["model"]["type"]
VOCAB = tj["model"]["vocab"]; UNK = tj["model"]["unk_id"]
with open(f"{OUT}/vocab.tsv", "w", encoding="utf-8") as f:
    for p, sc in VOCAB:
        f.write(p.replace("\t", " ").replace("\n", " ") + "\t" + repr(float(sc)) + "\n")
print("단어장", len(VOCAB), "개 / unk", UNK, "/ 정규화", tj.get("normalizer", {}).get("type"), "/ 앞처리", json.dumps(tj.get("pre_tokenizer"), ensure_ascii=False)[:200])
for f in ["minilm.onnx", "minilm_int8.onnx", "vocab.tsv"]:
    print(f, round(os.path.getsize(f"{OUT}/{f}") / 1e6, 1), "MB")
print("보드용 파일 준비 완료")

In [ ]:
# 8. ONNX가 원래 모델과 같은 벡터를 내는지 확인 + int8로 판단 결과가 바뀌는지
import unicodedata as ud
PIECES = {p: (i, float(sc)) for i, (p, sc) in enumerate(VOCAB)}
MAXLEN = min(32, max(len(p) for p, _ in VOCAB))
UNK_SCORE = min(float(sc) for _, sc in VOCAB) - 10.0
BOS, EOS = tok.cls_token_id, tok.sep_token_id
def norm(t):                      # 보드: NFKC + 공백 하나로
    return " ".join(ud.normalize("NFKC", t).split())
def viterbi(w):
    n = len(w); NEG = -1e30
    best = [NEG] * (n + 1); back = [None] * (n + 1); best[0] = 0.0
    for i in range(n):
        if best[i] <= NEG: continue
        single = False
        for L in range(1, min(MAXLEN, n - i) + 1):
            e = PIECES.get(w[i:i + L])
            if e is None: continue
            if L == 1: single = True
            v = best[i] + e[1]
            if v > best[i + L]: best[i + L] = v; back[i + L] = (i, e[0])
        if not single:            # 단어장에 없는 글자 → unk
            v = best[i] + UNK_SCORE
            if v > best[i + 1]: best[i + 1] = v; back[i + 1] = (i, UNK)
    ids, j = [], n
    while j > 0:
        i, pid = back[j]; ids.append(pid); j = i
    ids.reverse()
    out = []
    for x in ids:                 # 연속 unk는 하나로
        if not (x == UNK and out and out[-1] == UNK): out.append(x)
    return out
def ids_of(t):
    ids = []
    for w in norm(t).split(" "):
        if w: ids += viterbi("▁" + w)
    return [BOS] + ids + [EOS]
allt = train["text"].tolist() + evals["text"].tolist() + ["오늘 친구랑 싸웠어", "엄마 보고 싶어 ㅠㅠ", "1등 했어!", "ABC 학교"]
diff = [(t, ids_of(t), tok(t)["input_ids"]) for t in allt if ids_of(t) != tok(t)["input_ids"]]
print(f"토큰 비교 {len(allt)}문장: 다름 {len(diff)}")
for d in diff[:5]: print(d)
assert not [d for d in diff if all('가' <= c <= '힣' or c == ' ' for c in d[0])], "한글 문장에서 토큰이 달라요. 이 셀 출력 화면을 Claude에게 보내 주세요."
ADD_SPECIAL = True; TOK_IN, TOK_OUT = "kotlin-unigram", []
def onnx_embed(sess, texts):
    vs = []
    for t in texts:
        ids = np.array([ids_of(t)], dtype=np.int64)
        vs.append(sess.run(None, {"input_ids": ids, "attention_mask": np.ones_like(ids)})[0][0])
    return np.array(vs, dtype=np.float32)

sample = evals["text"].tolist()
for name in ["minilm.onnx", "minilm_int8.onnx"]:
    sess = ort.InferenceSession(f"{OUT}/{name}")
    V = onnx_embed(sess, sample)
    cos = (V * E).sum(1)
    sims = V @ A.T; best = sims.argmax(1)
    same = (anchor_id_arr[best] == evals.emb_id.values).mean()
    print(f"{name}: 원래 벡터와 코사인 평균 {cos.mean():.4f}(최소 {cos.min():.4f}) / 가장 가까운 기준 문장 번호 같음 {same*100:.1f}%")
    if name == "minilm_int8.onnx":
        E_int8, emb_int8 = V, sims[np.arange(len(V)), best]

In [ ]:
# 9. int8 모델로 임계값 다시 확인 (보드는 int8을 씀) → 기준 문장도 int8 벡터로 저장
sess8 = ort.InferenceSession(f"{OUT}/minilm_int8.onnx")
A8 = onnx_embed(sess8, anchor_text)
A_save = A8
sims8 = E_int8 @ A8.T; b8 = sims8.argmax(1)
evals["emb_id"], evals["emb_score"] = anchor_id_arr[b8], sims8[np.arange(len(E_int8)), b8]
A = A8
rows = []
for th in np.arange(0.40, 0.96, 0.05):
    p = pd.Series(combined(th), index=evals.index, dtype="object")
    right = (p[situ] == evals.gold[situ]).sum(); wrong = (p[situ].notna() & (p[situ] != evals.gold[situ])).sum(); false = p[~situ].notna().sum()
    rows.append(dict(임계값=round(th, 2), 상황맞힘=round(right / situ.sum() * 100, 1), 다른상황=int(wrong), 일반잘못잡음=int(false), 잘못재생률=round((wrong + false) / len(evals) * 100, 1)))
sweep8 = pd.DataFrame(rows); print(sweep8.to_string(index=False))
ok = sweep8[sweep8.잘못재생률 <= 5.0]
best8 = ok.sort_values(["상황맞힘", "임계값"], ascending=[False, False]).iloc[0] if len(ok) else sweep8.iloc[-1]
TH8 = float(best8.임계값)
print(f"\n보드(int8) 임계값 {TH8}: 상황 맞힘 {best8.상황맞힘}% / 다른 상황 {best8.다른상황} / 일반 잘못 잡음 {best8.일반잘못잡음}")

In [ ]:
# 10. 저장 + 다운로드 (zip 하나)
A_save.astype("<f4").tofile(f"{OUT}/anchors.f32")
meta = {"model": MODEL, "onnx": "minilm_int8.onnx", "dim": int(A_save.shape[1]), "count": int(A_save.shape[0]),
        "threshold": TH8, "tokenizer": "unigram", "vocab": "vocab.tsv", "unk": int(UNK),
        "unk_score": UNK_SCORE, "max_piece": MAXLEN, "bos": int(BOS), "eos": int(EOS), "rule": "키워드 없을 때만. 가장 가까운 기준 문장의 번호가 0이 아니고 코사인 ≥ threshold면 그 상황, 아니면 없음",
        "data": os.path.basename(XLSX), "ids": [int(i) for i in anchor_id], "texts": anchor_text}
json.dump(meta, open(f"{OUT}/anchors_meta.json", "w", encoding="utf-8"), ensure_ascii=False)
test = [{"text": t, "ids": ids_of(t), "vector_head": [round(float(x), 5) for x in v[:8]]} for t, v in zip(sample[:5], E_int8[:5])]
json.dump(test, open(f"{OUT}/embed_check.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)   # 보드 결과와 비교용
sweep8.to_csv(f"{OUT}/threshold_sweep.csv", index=False, encoding="utf-8-sig")
import shutil
os.remove(f"{OUT}/minilm.onnx")      # fp32는 크기만 커서 뺌
shutil.make_archive("/content/ongi_embed", "zip", OUT)
print(os.listdir(OUT), round(os.path.getsize("/content/ongi_embed.zip") / 1e6, 1), "MB")
try:
    from google.colab import files; files.download("/content/ongi_embed.zip")
except Exception as e:
    print("왼쪽 파일 탭에서 받기:", e)